# Infrastructure Overview\n\nThis notebook presents the overall platform architecture in one visual narrative for technical and non-technical audiences.\n

## Executive summary\n\nWe are building a **notebook-first ML platform reference** with:\n\n- **MLflow** as the system of record for experiments, runs, models, and lineage\n- **PostgreSQL + S3** as the MLflow persistence baseline (MinIO locally for parity)\n- **Lambda.ai + Slurm** for distributed training/inference coordination\n- **AWS + Kubernetes** for non-Lambda platform services and production operations\n- **python-terraform** for Python-managed infrastructure workflows\n- **Web UI + immutable notebook revisions** for controlled execution without notebook editing\n

## End-to-end architecture\n\n```mermaid\nflowchart LR\n    U[ML Engineer] --> UI[Notebook Web UI\\n(immutable revision selection)]\n    UI -->|submit execution request| ORCH[Execution Orchestrator\\n(contract + policy checks)]\n\n    subgraph LOCAL[Local parity environment]\n      LOC_RUN[Local runner]\n      LOC_PG[(PostgreSQL)]\n      LOC_S3[(MinIO / S3-compatible)]\n      LOC_K8S[Kubernetes-like local control plane]\n      LOC_SLRM[Slurm-like local scheduling path]\n    end\n\n    subgraph LAMBDA[Lambda.ai distributed compute]\n      SLRM[Slurm controller]\n      TRAIN[Distributed training / inference jobs]\n    end\n\n    subgraph AWS[AWS platform services]\n      K8S[EKS / Kubernetes services]\n      S3[(S3 artifacts)]\n      RDS[(RDS PostgreSQL)]\n    end\n\n    ORCH --> LOC_RUN\n    ORCH --> SLRM\n    ORCH --> K8S\n\n    LOC_RUN --> MLFLOW[MLflow Tracking + Registry]\n    TRAIN --> MLFLOW\n    K8S --> MLFLOW\n\n    MLFLOW --- LOC_PG\n    MLFLOW --- LOC_S3\n    MLFLOW --- RDS\n    MLFLOW --- S3\n\n    LOC_K8S -.local parity.-> K8S\n    LOC_SLRM -.local parity.-> SLRM\n\n    MLFLOW --> OBS[Observability + Cost\\n(Evidently, Prometheus, Grafana, AWS cost, Lambda attribution)]\n    OBS --> U\n```\n

## Audience views\n\n### For ML engineers\nYou choose an immutable notebook revision, trigger a run, and follow all outcomes in MLflow-linked visibility.\n\n### For platform engineers\nExecution contracts normalize requests across local, Slurm, and Kubernetes while preserving lineage/security constraints.\n\n### For leadership/stakeholders\nThis architecture prioritizes reproducibility, traceability, and operational realism from local validation to distributed production controls.\n

## Non-negotiable constraints represented\n\n1. Python-first, Linux-only, PyTorch + GPU posture\n2. Docker-first reproducibility, with Nix as helper layer\n3. MLflow with PostgreSQL backend and S3 artifact storage\n4. Lambda.ai Slurm for distributed coordination and failure handling\n5. AWS Kubernetes for non-Lambda platform services\n6. python-terraform as default infrastructure workflow\n7. Security, lineage, experiment/model traceability, and reproducibility as hard requirements\n